## Goal

1. Provide a list of properties with a set of fields such as size of building, age etc as input
2. Take a particular building and see if its price range is in the prediction range?

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

//anaconda/envs/uatu/lib/python3.7/site-packages/pandas/compat/_optional.py:138: UserWarning: Pandas requires version '2.7.0' or newer of 'numexpr' (version '2.6.9' currently installed).
  warnings.warn(msg, UserWarning)
//anaconda/envs/uatu/lib/python3.7/site-packages/sklearn/linear_model/least_angle.py:30: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  method='lar', copy_X=True, eps=np.finfo(np.float).eps,
//anaconda/envs/uatu/lib/python3.7/site-packages/sklearn/linear_model/least_angle.py:167: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and

## Load price data

In [2]:
# fn = "../data/multifamily_prices.tsv"
# fn = "../data/multifamily_prices_dec_27_2021.tsv"
fn = "../data/multifamily_prices_jan_04_2022.tsv"

df = pd.read_csv(fn, sep="\t")

## Review data

In [3]:
df.head(20)

,Address,size,lot size,bedrooms,bathroom,built year,Age,units,address,address category,Ask Price,Link,Sold Price
0,25-27 Morrell,1888,NaN,4,2.0,1923.0,98.0,2.0,nobhill,nobhill,1750000,https://www.redfin.com/CA/San-Francisco/25-Mor...,NaN
1,1133 - 1135 Judah St,2472,NaN,4,2.0,1946.0,75.0,2.0,sunset,sunset,1698000,https://www.redfin.com/CA/San-Francisco/1133-J...,NaN
2,1572-1576 Bush St,5113,1754.0,12,4.0,1907.0,114.0,NaN,eastbush,eastbush,1750000,https://www.redfin.com/CA/San-Francisco/1572-B...,NaN
3,500 - 502 Andover St,2658,NaN,4,2.0,1963.0,58.0,NaN,bernalheights,mission,1795000,https://www.redfin.com/CA/San-Francisco/500-An...,NaN
4,916 - 920 Clay St,3681,1337.0,12,2.0,1907.0,114.0,3.0,chinatown,nobhill,1675000,https://www.redfin.com/CA/San-Francisco/916-Cl...,NaN
5,51 - 55 Webster St,3069,1947.0,6,3.5,1900.0,121.0,3.0,hayesvalley,center,1995000,https://www.redfin.com/CA/San-Francisco/51-Web...,NaN
6,1012 Page St,3780,NaN,7,4.0,1908.0,113.0,NaN,lowerhaight,center,1595000,https://www.redfin.com/CA/San-Francisco/1012-P...,NaN
7,437 - 443 Austin St,5661,1873.0,12,2.0,1900.0,121.0,4.0,eastbush,eastbush,2000000,https://www.redfin.com/CA/San-Francisco/437-Au...,2000000.0
8,"1127 Filbert St, San Francisco, CA 94109",3329,1350.0,5,4.5,1929.0,92.0,3.0,nobhill,nobhill,2850000,https://www.redfin.com/CA/San-Francisco/1127-F...,NaN
9,"630 - 632 Anza St, San Francisco, CA 94118",2070,940.0,5,2.0,1932.0,89.0,2.0,sunset,sunset,2098000,https://www.redfin.com/CA/San-Francisco/630-An...,NaN


In [4]:
print("No of rows in price table: {}".format(df.shape[0]))

assert df.shape[0] > 10

No of rows in price table: 13


## Take out a given building

In [5]:
# 1059 Montgomery St,
take_out_inx = 12
takeout_item_no = 1

## Exctract features

In [6]:
# Extract size of the building
mylist = df['size'].tolist()
x_size = mylist[:take_out_inx] + mylist[take_out_inx+1:]

# Extract no of bedrooms
mylist = df['bedrooms'].tolist()
x_bed = mylist[:take_out_inx] + mylist[take_out_inx+1:]

# Extract no of bathrooms
mylist = df['bathroom'].tolist()
x_bath = mylist[:take_out_inx] + mylist[take_out_inx+1:]

# Extract age of building
mylist = df['Age'].tolist()
x_age = mylist[:take_out_inx] + mylist[take_out_inx+1:]

# Extract address category column
mylist = df['address category'].tolist()
x_address = mylist[:take_out_inx] + mylist[take_out_inx+1:]

# Extract Ask price
mylist = df['Ask Price'].tolist()
y_ = mylist[:take_out_inx] + mylist[take_out_inx+1:]


## Test features size

In [8]:
assert len(x_size) == df.shape[0] - takeout_item_no

assert len(x_bed) == df.shape[0] - takeout_item_no

assert len(x_bath) == df.shape[0] - takeout_item_no

assert len(x_age) == df.shape[0] - takeout_item_no

assert len(x_address) == df.shape[0] - takeout_item_no

assert len(y_) == df.shape[0] - takeout_item_no

## Handle NAN feature values!

In [9]:
# Replace nan with mean
def impute(fet):
    impued_fet = []
    sum_, cnt = 0, 0
    nan_inxs = []
    for inx, x in enumerate(fet):
        if not np.isnan(x):
            impued_fet.append(x)
            sum_ += x
            cnt += 1
        else:
            nan_inxs.append(inx)
    avg = sum_/cnt
    for inx in nan_inxs:
        fet[inx] = avg
    
    return fet

## Impute all numerical feature

In [10]:
imputed_x_size = impute(x_size)
imputed_x_bed = impute(x_bed)
imputed_x_bath = impute(x_bath)
imputed_x_age = impute(x_age)
imputed_y_ = impute(y_)

In [11]:
print("Size of imputed_x_size: {}".format(len(imputed_x_size)))

print("Size of imputed_y_: {}".format(len(imputed_y_)))

Size of imputed_x_size: 12
Size of imputed_y_: 12


## Handle categorical feature

In [14]:
import pandas as pd

data = pd.DataFrame({'address': x_address})
address_fets_df = pd.get_dummies(data)

In [19]:
address_fets_df.head(20)

,address_center,address_eastbush,address_mission,address_nobhill,address_sunset
0,0,0,0,1,0
1,0,0,0,0,1
2,0,1,0,0,0
3,0,0,1,0,0
4,0,0,0,1,0
5,1,0,0,0,0
6,1,0,0,0,0
7,0,1,0,0,0
8,0,0,0,1,0
9,0,0,0,0,1


## Extract address features

In [17]:
x_address_center = address_fets_df.address_center.to_list()
x_address_eastbush = address_fets_df.address_eastbush.to_list()
x_address_mission = address_fets_df.address_mission.to_list()
x_address_nobhill = address_fets_df.address_nobhill.to_list()
x_address_sunset = address_fets_df.address_sunset.to_list()

In [18]:
x_address_nobhill

[1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0]

In [30]:
len(x_address_nobhill)

12

In [31]:
len(x_size)

12

## Normalize numerical features

In [53]:
max_x_size = max(x_size)
x_size_norm = [1.0*x/max_x_size for x in x_size]

max_x_bed = max(x_bed)
x_bed_norm = [1.0*x/max_x_bed for x in x_bed]

max_x_bath = max(x_bath)
x_bath_norm = [1.0*x/max_x_bath for x in x_bath]

max_x_age = max(x_age)
x_age_norm = [1.0*x/max_x_age for x in x_age]

max_y_ = max(y_)
y_norm_ = [1.0*x/max_y_ for x in y_]

## Review features

In [54]:
print("Size of y_norm: {}".format(len(y_norm_)))

Size of y_norm: 12


In [55]:
y_norm_

[0.6140350877192983,
 0.5957894736842105,
 0.6140350877192983,
 0.6298245614035087,
 0.5877192982456141,
 0.7,
 0.5596491228070175,
 0.7017543859649122,
 1.0,
 0.736140350877193,
 0.631578947368421,
 0.6842105263157895]

In [56]:
y_

[1750000,
 1698000,
 1750000,
 1795000,
 1675000,
 1995000,
 1595000,
 2000000,
 2850000,
 2098000,
 1800000,
 1950000]

## Include address?

It seems including address turn off predictions a lot!!

In [42]:
include_address = False

## Construct X & y numpy arrays

In [57]:
X = np.array([x_size, x_bed, x_bath, x_age])

if include_address:
    # Add one-hot encoded of address
    X = np.array([x_size, x_bed, x_bath, x_age, x_address_center, x_address_eastbush, 
                       x_address_mission, x_address_nobhill, x_address_sunset])

X_norm = np.array([x_size_norm, x_bed_norm, x_bath_norm, x_age_norm])
if include_address:
    # Add one-hot encoded address
    X_norm = np.array([x_size_norm, x_bed_norm, x_bath_norm, x_age_norm, x_address_center, x_address_eastbush, 
                   x_address_mission, x_address_nobhill, x_address_sunset])

X = np.transpose(X)
X_norm = np.transpose(X_norm)

y = np.transpose(np.array([y_]))
y_norm = np.transpose(np.array([y_norm_]))

## Review

In [44]:
X.shape

(12, 4)

In [45]:
y.shape

(12, 1)

In [46]:
X_norm.shape

(12, 4)

In [58]:
y_norm.shape

(12, 1)

In [47]:
assert X.shape[0] == y.shape[0]

In [59]:
assert X_norm.shape[0] == y_norm.shape[0]

## Train two Regression model!

In [60]:
# Train a model wo normalized feature
reg = LinearRegression().fit(X, y)

# Train a model using normalized feature values
reg_norm = LinearRegression().fit(X_norm, y_norm)

## Review models coeffs

In [61]:
reg.coef_

array([[   131.39388905, -59184.06762331,  77519.67164974,
          -837.49316475]])

In [62]:
reg.intercept_

array([1739645.44991144])

In [63]:
reg_norm.coef_

array([[ 0.26098976, -0.24919607,  0.13599942, -0.03555673]])

In [64]:
reg_norm.intercept_

array([0.61040191])

## Predict Taken out building ASK Price!

In [68]:
# Morrel building
# x_morrel = np.array([[1888, 4, 2.0, 98]])
# x_morrel_norm = np.array([[x_morrel[0][0]/max_x_size, x_morrel[0][1]/max_x_bed, x_morrel[0][2]/max_x_bath, x_morrel[0][3]/max_x_age]])

# y_pred_1 = reg.predict(x_morrel)
# y_pred_2 = reg_2.predict(x_morrel_norm)

# Montgomery
x_montgomery = np.array([[1600, 3, 2.0, 114]])
if include_address:
    x_montgomery = np.array([[1600, 3, 2.0, 114, 0, 0, 0, 1, 0]])
x_montgomery_norm = np.array([[x_montgomery[0][0]/max_x_size, x_montgomery[0][1]/max_x_bed, x_montgomery[0][2]/max_x_bath, x_montgomery[0][3]/max_x_age]])

y_pred_1 = reg.predict(x_montgomery)
y_pred_2 = reg_norm.predict(x_montgomery_norm)


## Print Predicted ASK PRice!!

In [69]:
y_pred_1

array([[1831888.59203848]])

In [70]:
y_pred_2*max_y_

array([[1831888.59203848]])

## How much underprice this building is?

In [34]:
# Morrel
# ask_price = 1750000
# prediction = 1871527.54146896

# Montgomery
ask_price = 1498000
prediction = y_pred_2[0][0]*max_y_

print("Prediction: {}".format(prediction))
print("Discount percentage: {}".format((prediction-ask_price)/prediction))

Prediction: 1831888.5920384827
Discount percentage: 0.18226468219169337


## Simulate offer discounts

In [62]:
offer_0 = 1800000
offer_1 = 1700000
offer_2 = 1690000
offer_3 = 1680000
offer_4 = 1675000
offer_5 = 1670000
offer_6 = 1650000

In [63]:
(ask_price-offer_0)/ask_price

-0.02857142857142857

In [64]:
(ask_price-offer_1)/ask_price

0.02857142857142857

In [104]:
(ask_price-offer_2)/ask_price

0.03428571428571429

In [105]:
(ask_price-offer_3)/ask_price

0.04

In [106]:
(ask_price-offer_4)/ask_price

0.04285714285714286

In [107]:
(ask_price-offer_5)/ask_price

0.045714285714285714

In [108]:
(ask_price-offer_6)/ask_price

0.05714285714285714

In [112]:
1750000-72000

1678000

In [113]:
(ask_price-1670000)/ask_price

0.045714285714285714